In [19]:
import pandas as pd

df = pd.read_csv("data/all_rows_merged.csv")
df

,Probe,Lab_Number,Prediction,Overall_Assessment_pred_file,Overall_Interpretation,Overall_Assessment_data
0,1,P1300010,0,1,Poziom metalicznych produktów zużycia w normie...,WSKAZÓWKA
1,1,P1304884,0,0,Poziom metalicznych produktów zużycia w normie...,W NORMIE
2,1,P1400003,2,2,W efekcie zużycia i/lub korozji poziom miedzi ...,UWAGA
3,1,P1400044,2,2,"P1400044 - próbka oleju z filtra, P1400003 - p...",UWAGA
4,1,P1400113,0,0,NaN,W NORMIE
...,...,...,...,...,...,...
18843,test,P2502080,1,1,Lepkość poniżej zakresu typowego dla klasy SAE...,WSKAZÓWKA
18844,test,P2502129,2,2,Oznaczona lepkość na niskim poziomie. Widmo po...,UWAGA
18845,test,P2505955,0,0,Oznaczona lepkość dla klasy SAE 40 w górnym za...,W NORMIE
18846,test,P2505960,0,0,Oznaczona lepkość dla klasy SAE 40 w normie. T...,W NORMIE


In [20]:
# Wywołanie parsera i zapisanie pliku JSON

from parser import parser
import json

data = parser.df_to_json(df, 0, len(df))

with open("data/parsed_samples.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

In [21]:
from parser.patterns import FEATURE_TO_PATTERN, STATUS_TO_ABNORMALITY

quantified_dict, quantified_df = parser.parser_quantifier(
    parser_output="data/parsed_samples.json",
    feature_to_pattern=FEATURE_TO_PATTERN,
    status_to_abnormality=STATUS_TO_ABNORMALITY,
    return_df=True,
    fill_missing=0,   # 0 oznacza że brak wzmianki to norma
    keep_none=True 
)

In [22]:
quantified_df.to_csv("data/parsed_samples_all_quantified.csv", encoding="utf-8")

In [23]:
import random
import json

# tylko te laby, które są w data i mają 2 lub mniej kluczy w wyniku parsera
labs = [str(x) for x in df["Lab_Number"].tolist()]
available_labs = [
    lab for lab in labs
    if lab in data and isinstance(data[lab], dict) and len(data[lab]) <= 2
]

k = min(50, len(available_labs))
sample_labs = random.sample(available_labs, k)

for lab in sample_labs:
    idx = df.index[df["Lab_Number"].astype(str) == lab][0]
    inp = df.loc[idx, "Overall_Interpretation"]

    print("=" * 120)
    print(f"Lab_Number: {lab}")
    print("- INPUT (Overall_Interpretation) -")
    print("" if inp is None else str(inp))
    print("- OUTPUT (parsed) -")
    print(json.dumps({lab: data[lab]}, ensure_ascii=False, indent=2))

Lab_Number: P2102280
- INPUT (Overall_Interpretation) -
Wszystkie oznaczone parametry w normie.

Wnioski i Zalecenia:

Obserwować zmiany w następnej analizie. Przeprowadzić kolejne badanie zgodnie z przyjętym harmonogramem diagnostyki płynu.
- OUTPUT (parsed) -
{
  "P2102280": {
    "oznaczone_parametry": "w_normie"
  }
}
Lab_Number: P1702770
- INPUT (Overall_Interpretation) -
Dla danej próbki oznaczyliśmy wymagane parametry. 

Wnioski i zalecenia:  
Powiązać wykryte metale zużyciowe z metalurgią układu olejowego W celu obserwacji trendu proszę wykonać badanie kolejnej próbki zgodnie z przyjętym harmonogramem diagnostyki olejowej.
- OUTPUT (parsed) -
{
  "P1702770": {}
}
Lab_Number: P2011758
- INPUT (Overall_Interpretation) -
Oznaczona lepkość, poziom dodatków uszlachetniających oraz poziom metalicznych produktów zużycia oznaczony w badaniu pierwiastków (cząstki o rozmiarze <5um) w normie. 

Zalecenia i wnioski:

Proszę obserwować zmiany w następnej analizie. Przeprowadzić kolejne bada

In [24]:
import rules

false_negatives = rules.get_false_negatives(data, df, min_negative_params=2)

false_positives = rules.get_false_positives(data, df)

potential_guidelines = rules.get_potential_guidelines(
    data, df,
    suspicious_classes=(0, 2),
    min_borderline_params=1,
    min_total_params=3,
    ok_to_borderline_ratio=1.0,
    require_softening_phrase=False,
)

print("false_positives: ", len(false_positives), false_positives)
print("false_negatives_3: ", len(false_negatives), false_negatives)
print("potential_guidelines: ", len(potential_guidelines), potential_guidelines)

false_positives:  653 ['P1300010', 'P1401076', 'P1401180', 'P1402230', 'P1501440', 'P1502422', 'P1502980', 'P1503769', 'P1600024', 'P1601705', 'P1602482', 'P1602839', 'P1603207', 'P1603214', 'P1603284', 'P1603286', 'P1603294', 'P1603477', 'P1603489', 'P1603491', 'P1603526', 'P1604035', 'P1604043', 'P1604091', 'P1604119', 'P1604235', 'P1604287', 'P1604393', 'P1604495', 'P1604533', 'P1604583', 'P1604677', 'P1604968', 'P1605410', 'P1605427', 'P1605620', 'P1605631', 'P1605632', 'P1605638', 'P1605642', 'P1700327', 'P1700357', 'P1700591', 'P1700691', 'P1700872', 'P1700874', 'P1701267', 'P1701397', 'P1701411', 'P1701436', 'P1701437', 'P1701637', 'P1701926', 'P1702107', 'P1702571', 'P1702579', 'P1702773', 'P1703054', 'P1703620', 'P1704085', 'P1704300', 'P1704347', 'P1704374', 'P1704375', 'P1705390', 'P1705922', 'P1705923', 'P1800048', 'P1800196', 'P1800375', 'P1800418', 'P1800422', 'P1800645', 'P1800646', 'P1800647', 'P1800780', 'P1801492', 'P1801986', 'P1802216', 'P1802574', 'P1803173', 'P180

In [25]:
from utils import labs_to_review_json, labs_to_review_csv

review = labs_to_review_json(false_positives, df, data)

# save:
# with open("false_positives.json", "w", encoding="utf-8") as f:
#     json.dump(review, f, indent=2, ensure_ascii=False)

filtered_review = {
    lab: {k: v for k, v in vals.items() if k != "prediction"}
    for lab, vals in review.items()
}

print(json.dumps(filtered_review, indent=2, ensure_ascii=False))

{
  "P1300010": {
    "sample": "Poziom metalicznych produktów zużycia w normie,  świadczy o braku intensywnych procesów zużyciowych oraz korozyjnych w układzie olejowym. Oznaczona lepkość w normie. Liczba zasadowa znajduje się na bezpiecznym poziomie. \n\nZalecenia i wnioski:\nPrzy zachowaniu zbliżonych warunków eksploatacyjnych oraz czynności obsługowych, olej kwalifikuje się do dalszej eksploatacji bez podejmowania dodatkowych czynności korygujących. W celu obserwacji zmiany trendu przesłać próbkę do badania po kolejnych 250rbh.",
    "parser_result": {
      "metale_zużyciowe": "w_normie",
      "lepkość": "w_normie",
      "liczba_zasadowa": "w_zakresie_bezpiecznym"
    },
    "ground_true": "WSKAZÓWKA"
  },
  "P1401076": {
    "sample": "Poziom fosforu podwyższony, w zakresie granicznym. Poziom cynku powyżej dopuszczalnej granicy. Pozostałe oznaczenia w znajdują się w dopuszczalnym przedziale. Zalecenia i wnioski: Poziomy podwyższone cynku i fosforu wskazują na zastosowanie na do

In [26]:

# create dataframe
review_df = labs_to_review_csv(false_positives, df, data)
review_df.to_csv("samples_to_exclude_by_reason/false_positives.csv", index=False, encoding="utf-8")

review_df = labs_to_review_csv(false_negatives, df, data)
review_df.to_csv("samples_to_exclude_by_reason/false_negatives.csv", index=False, encoding="utf-8")

review_df = labs_to_review_csv(potential_guidelines, df, data)
review_df.to_csv("samples_to_exclude_by_reason/potential_guidelines.csv", index=False, encoding="utf-8")